## Environment & folder setup
This code cell installs the necessary Python packages for a machine learning project, specifically focusing on libraries commonly used for deep learning with PyTorch (torch, torchvision, torchaudio), transformers and datasets (transformers, datasets), and evaluation tools (evaluate, accelerate, opacus, scikit-learn, pandas). It also creates essential directories (data, models, results, Figures, scripts) to organize project files.

In [1]:
# env
!pip install torch torchvision torchaudio  # pick CUDA wheel if you have GPU
!pip install transformers datasets accelerate evaluate opacus scikit-learn pandas
!pip install datasets


# folders (align with your project)
!mkdir -p /kaggle/working/dataset /kaggle/working/llm_models /kaggle/working/llm_results /kaggle/working/llm_figures /kaggle/working/llm_scripts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 72.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

## Imports

In [2]:
import csv
import os, math, json
import numpy as np
import pandas as pd
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, recall_score

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from opacus import PrivacyEngine
from opacus.utils.uniform_sampler import UniformWithReplacementSampler
from opacus.accountants.utils import get_noise_multiplier

from datasets import load_dataset

## Config

In [3]:
# ======================
# Config
# ======================
@dataclass
class Config:
    model_name: str = "distilbert-base-uncased"
    max_len: int = 256
    train_frac: float = 0.8
    valid_frac: float = 0.1  # remaining 0.1 is test
    batch_size: int = 32
    epochs: int = 5
    lr: float = 5e-5
    delta: float = 1e-5
    max_grad_norm: float = 1.0
    noise_multiplier: float = 0.5  # try {0.5, 0.8, 1.0, 1.2}
    freeze_encoder: bool = True
    seed: int = 42
    save_dir: str = "/kaggle/working/llm_models/distilbert_dp_cls"
    logs_csv: str = "/kaggle/working/llm_results/llm_dp_metrics.csv"
    temporal_logs_csv: str = "/kaggle/working/llm_results/llm_dp_metrics_temporal.csv"
    data_csv: str = "/kaggle/working/dataset/enron_spam_data.csv"

cfg = Config()

## Reproducibility

In [4]:
# ======================
# Reproducibility
# ======================
def set_seed(seed: int):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load & Split Data

In [5]:
# ======================
# Load & split data
# ======================
# load enron spam from HF
raw = load_dataset("SetFit/enron_spam")

# inspect
print(raw)
print(raw["train"][0])

# create splits: 80/10/10
split = raw["train"].train_test_split(test_size=0.2, seed=42)
tmp   = split["test"].train_test_split(test_size=0.5, seed=42)

dataset = {
    "train": split["train"],
    "validation": tmp["train"],
    "test": tmp["test"]
}

print(dataset)


README.md:   0%|          | 0.00/176 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/101M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/6.27M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/31716 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date'],
        num_rows: 31716
    })
    test: Dataset({
        features: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date'],
        num_rows: 2000
    })
})
{'message_id': 33214, 'text': 'any software just for 15 $ - 99 $ understanding oem software\nlead me not into temptation ; i can find the way myself .\n# 3533 . the law disregards trifles .', 'label': 1, 'label_text': 'spam', 'subject': 'any software just for 15 $ - 99 $', 'message': 'understanding oem software\nlead me not into temptation ; i can find the way myself .\n# 3533 . the law disregards trifles .', 'date': datetime.datetime(2005, 6, 18, 0, 0)}
{'train': Dataset({
    features: ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date'],
    num_rows: 25372
}), 'validation': Dataset({
    features: ['message_id', 'text', 'label', 'label_text', 'subject', 'm

## Tokenizer & Model

In [6]:
# ======================
# Tokenizer & model
# ======================
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

def tok_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=cfg.max_len
    )


tokenized = {}
for split_name in ["train","validation","test"]:
    tokenized[split_name] = dataset[split_name].map(tok_fn, batched=True, remove_columns=["text"])
    tokenized[split_name].set_format(type="torch", columns=["input_ids","attention_mask","label"])

model = AutoModelForSequenceClassification.from_pretrained(cfg.model_name, num_labels=2)
print("num_labels:", model.config.num_labels)
assert model.config.num_labels == 2


# Optionally freeze encoder for DP stability
if cfg.freeze_encoder:
    for n, p in model.named_parameters():
        if "classifier" not in n:
            p.requires_grad = False

model.to(device)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/25372 [00:00<?, ? examples/s]

...

2025-09-01 23:13:34.795346: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756768414.961722      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756768415.017899      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


num_labels: 2


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


## DP Sampler and Loaders

In [7]:
# ======================
# DP Sampler & loaders
# ======================
train_ds = tokenized["train"]
val_ds   = tokenized["validation"]
test_ds  = tokenized["test"]

sample_rate = cfg.batch_size / len(train_ds)

train_loader = DataLoader(
    train_ds,
    batch_sampler=UniformWithReplacementSampler(
        num_samples=len(train_ds),
        sample_rate=sample_rate,
        generator=torch.Generator().manual_seed(cfg.seed),
    ),
    num_workers=2,
)
val_loader  = DataLoader(val_ds,  batch_size=128, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2)

## Optimizer, loss, DP

In [8]:
# ======================
# Optimizer, loss, DP
# ======================
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=cfg.lr)
criterion = nn.CrossEntropyLoss()

privacy_engine = PrivacyEngine(accountant="rdp")
model.train()
model, optimizer, train_loader = privacy_engine.make_private(
    module=model,
    optimizer=optimizer,
    data_loader=train_loader,
    noise_multiplier=cfg.noise_multiplier,
    max_grad_norm=cfg.max_grad_norm,
    grad_sample_mode="hooks",
)

/usr/local/lib/python3.11/dist-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


## Evaluation Helper

In [9]:
# ======================
# Eval helper
# ======================
def evaluate(loader):
    model.eval()
    preds, gold = [], []
    with torch.no_grad():
        for batch in loader:
            inputs = {
                "input_ids": batch["input_ids"].to(device),
                "attention_mask": batch["attention_mask"].to(device),
            }
            labels = batch["label"].to(device).long()
            logits = model(**inputs).logits
            yhat = torch.argmax(logits, dim=1)
            preds.extend(yhat.cpu().numpy())
            gold.extend(labels.cpu().numpy())
    acc = accuracy_score(gold, preds)
    f1  = f1_score(gold, preds)
    rec = recall_score(gold, preds)
    return acc, f1, rec

## Train loop

In [10]:
# ======================
# Train loop
# ======================
os.makedirs(os.path.dirname(cfg.logs_csv), exist_ok=True)
logs = []
for epoch in range(1, cfg.epochs+1):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader, start=1):
        inputs = {
            "input_ids": batch["input_ids"].to(device),
            "attention_mask": batch["attention_mask"].to(device),
        }
        labels = batch["label"].to(device).long()

        optimizer.zero_grad()
        logits = model(**inputs).logits
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # log every 50 steps
        if step % 50 == 0:
            print(f"[Epoch {epoch}] Step {step}/{len(train_loader)} | Batch loss={loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    eps = privacy_engine.get_epsilon(delta=cfg.delta)
    val_acc, val_f1, val_rec = evaluate(val_loader)
    print(f"Epoch {epoch}/{cfg.epochs} | "
          f"avg train loss={avg_loss:.4f} | "
          f"ε={eps:.2f}, δ={cfg.delta} | "
          f"val acc={val_acc:.4f}, f1={val_f1:.4f}, rec={val_rec:.4f}")

    logs.append({
        "epoch": epoch,
        "epsilon": eps,
        "delta": cfg.delta,
        "val_accuracy": val_acc,
        "val_f1": val_f1,
        "val_recall": val_rec,
        "noise_multiplier": cfg.noise_multiplier,
        "max_grad_norm": cfg.max_grad_norm,
        "batch_size": cfg.batch_size,
        "lr": cfg.lr,
        "freeze_encoder": cfg.freeze_encoder,
    })

    log_entry = {
                "epoch": epoch,
                "epsilon": eps,
                "delta": cfg.delta,
                "avg_loss": avg_loss,
                "val_accuracy": val_acc,
                "val_f1": val_f1,
                "val_recall": val_rec
                }

    with open(cfg.temporal_logs_csv, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=log_entry.keys())
        if epoch == 1:  # write header only once
            writer.writeheader()
        writer.writerow(log_entry)

# Final test evaluation
test_acc, test_f1, test_rec = evaluate(test_loader)
print(f"[TEST] acc={test_acc:.4f}, f1={test_f1:.4f}, rec={test_rec:.4f}")

# Final results
print("=== Final Test Evaluation ===")
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test F1:       {test_f1:.4f}")
print(f"Test Recall:   {test_rec:.4f}")

logs.append({
    "epoch": "TEST",
    "epsilon": privacy_engine.get_epsilon(delta=cfg.delta),
    "delta": cfg.delta,
    "val_accuracy": None,
    "val_f1": None,
    "val_recall": None,
    "test_accuracy": test_acc,
    "test_f1": test_f1,
    "test_recall": test_rec,
    "noise_multiplier": cfg.noise_multiplier,
    "max_grad_norm": cfg.max_grad_norm,
    "batch_size": cfg.batch_size,
    "lr": cfg.lr,
    "freeze_encoder": cfg.freeze_encoder,
})

pd.DataFrame(logs).to_csv(cfg.logs_csv, index=False)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[Epoch 1] Step 50/791 | Batch loss=0.6990
[Epoch 1] Step 100/791 | Batch loss=0.6969
[Epoch 1] Step 150/791 | Batch loss=0.7003
[Epoch 1] Step 200/791 | Batch loss=0.6884
[Epoch 1] Step 250/791 | Batch loss=0.6857
[Epoch 1] Step 300/791 | Batch loss=0.6835
[Epoch 1] Step 350/791 | Batch loss=0.6433
[Epoch 1] Step 400/791 | Batch loss=0.6996
[Epoch 1] Step 450/791 | Batch loss=0.6587
[Epoch 1] Step 500/791 | Batch loss=0.7056
[Epoch 1] Step 550/791 | Batch loss=0.6583
[Epoch 1] Step 600/791 | Batch loss=0.6769
[Epoch 1] Step 650/791 | Batch loss=0.6766
[Epoch 1] Step 700/791 | Batch loss=0.6425
[Epoch 1] Step 750/791 | Batch loss=0.6246


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Epoch 1/5 | avg train loss=0.6754 | ε=4.63, δ=1e-05 | val acc=0.5183, f1=0.6826, rec=1.0000


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


...

[Epoch 5] Step 50/791 | Batch loss=0.4783
[Epoch 5] Step 100/791 | Batch loss=0.5500
[Epoch 5] Step 150/791 | Batch loss=0.4881
[Epoch 5] Step 200/791 | Batch loss=0.5143
[Epoch 5] Step 250/791 | Batch loss=0.4461
[Epoch 5] Step 300/791 | Batch loss=0.4029
[Epoch 5] Step 350/791 | Batch loss=0.5446
[Epoch 5] Step 400/791 | Batch loss=0.4419
[Epoch 5] Step 450/791 | Batch loss=0.4479
[Epoch 5] Step 500/791 | Batch loss=0.4008
[Epoch 5] Step 550/791 | Batch loss=0.3673
[Epoch 5] Step 600/791 | Batch loss=0.4180
[Epoch 5] Step 650/791 | Batch loss=0.5893
[Epoch 5] Step 700/791 | Batch loss=0.3609
[Epoch 5] Step 750/791 | Batch loss=0.3836


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Epoch 5/5 | avg train loss=0.4960 | ε=5.96, δ=1e-05 | val acc=0.7900, f1=0.8302, rec=0.9909


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[TEST] acc=0.7784, f1=0.8200, rec=0.9877
=== Final Test Evaluation ===
Test accuracy: 0.7784
Test F1:       0.8200
Test Recall:   0.9877


## Save Model

In [11]:
# ======================
# Save model cleanly
# ======================
os.makedirs(cfg.save_dir, exist_ok=True)
# Access the underlying model from the GradSampleModule
unwrapped_model = model._module
sd = unwrapped_model.state_dict()
clean_sd = {k.replace("_module.", ""): v for k, v in sd.items()}
unwrapped_model.save_pretrained(cfg.save_dir, state_dict=clean_sd)
tokenizer.save_pretrained(cfg.save_dir)

with open(os.path.join(cfg.save_dir, "train_config.json"), "w") as f:
    json.dump(cfg.__dict__, f, indent=2)

print(f"[DONE] Saved DP model + tokenizer to {cfg.save_dir}")
print(f"[DONE] Logged metrics to {cfg.logs_csv}")

[DONE] Saved DP model + tokenizer to /kaggle/working/llm_models/distilbert_dp_cls
[DONE] Logged metrics to /kaggle/working/llm_results/llm_dp_metrics.csv
